# Training and Comparing Neural Network Models

## Architectures:
1. LSTM - baseline recurrent network
2. GRU - simplified LSTM alternative
3. CNN+BiLSTM - hybrid architecture

## Dependency Installation and Imports

In [ ]:
# Install required packages
%pip install -q torch torchvision torchaudio
%pip install -q scikit-learn matplotlib seaborn
%pip install -q onnx onnxruntime
%pip install -q onnxscript

In [ ]:
import onnx
import json
import time
import torch
import shutil
import warnings
import numpy as np
import pandas as pd
import torch.nn as nn
import seaborn as sns
from pathlib import Path
import onnxruntime as ort
import torch.optim as optim
from google.colab import files, drive
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, average_precision_score, auc
)


In [ ]:
warnings.filterwarnings('ignore')

# Display setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device in use: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Number of features and window length
FEATURES = 32
SEQ_LEN  = 10

# Seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    
# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

## Load data

In [ ]:
# Data folder from Google Drive
DATA_PATH = Path('/content/drive/MyDrive/Colab Notebooks')
DATA_PATH.mkdir(exist_ok=True)

# Load data
print('\nLoad data...')
X_train = np.load(DATA_PATH / 'X_train.npy')
y_train = np.load(DATA_PATH / 'y_train.npy')
X_val = np.load(DATA_PATH / 'X_val.npy')
y_val = np.load(DATA_PATH / 'y_val.npy')
X_test = np.load(DATA_PATH / 'X_test.npy')
y_test = np.load(DATA_PATH / 'y_test.npy')

# Load metadata
with open(DATA_PATH / 'feature_names.json', 'r') as f:
    feature_names = json.load(f)
with open(DATA_PATH / 'file_names.json', 'r') as f:
    file_names = json.load(f)
file_boundaries = np.load(DATA_PATH / 'file_boundaries.npy')

print(f'\nData shapes:')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}, y_val: {y_val.shape}')
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')


In [ ]:
# Class-balance analysis
train_class_dist = np.bincount(y_train.astype(int))
print(f'\nClass distribution in train:')
print(f'Class 0 (normal mode): {train_class_dist[0]} ({train_class_dist[0]/len(y_train)*100:.1f}%)')
print(f'Class 1 (fault mode): {train_class_dist[1]} ({train_class_dist[1]/len(y_train)*100:.1f}%)')

# Calculate class weights for the loss function
class_weights = len(y_train) / (2 * train_class_dist)
class_weights = torch.FloatTensor(class_weights).to(device)
print(f'\nClass weights for loss: {class_weights.cpu().numpy()}')

In [ ]:
# Add noise to test data
noise_level = 0.05
X_train = X_train + np.random.normal(0, noise_level, X_train.shape)
X_test = X_test + np.random.normal(0, noise_level, X_test.shape)
X_val = X_val + np.random.normal(0, noise_level, X_val.shape)

## Preparing PyTorch Dataset and DataLoader

In [ ]:
class TimeSeriesDataset(Dataset):
    """Dataset for time series"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset = TimeSeriesDataset(X_val, y_val)
test_dataset = TimeSeriesDataset(X_test, y_test)

# Training parameters
BATCH_SIZE = 256
NUM_WORKERS = 0

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

## Model Architectures

In [ ]:
class LSTMModel(nn.Module):
    """LSTM architecture for time-series classification"""
    def __init__(self, input_size=FEATURES, hidden_size=32, num_layers=2, dropout=0.3):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout1 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch, SEQ_LEN=10, features=46)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use the last timestep
        last_hidden = lstm_out[:, -1, :]  # (batch, hidden_size)

        out = self.dropout1(last_hidden)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout2(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out.squeeze(-1)

In [ ]:
class GRUModel(nn.Module):
    """GRU architecture for time-series classification"""
    def __init__(self, input_size=FEATURES, hidden_size=32, num_layers=2, dropout=0.3):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout1 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        gru_out, h_n = self.gru(x)
        last_hidden = gru_out[:, -1, :]

        out = self.dropout1(last_hidden)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout2(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out.squeeze(-1)


In [ ]:
# class GRUModel(nn.Module):
# English note.
#     def __init__(self, input_size=FEATURES, hidden_size=32, num_layers=2, dropout=0.3):
#         super(GRUModel, self).__init__()
#         self.gru = nn.GRU(
#             input_size=input_size,
#             hidden_size=hidden_size,
#             num_layers=num_layers,
#             batch_first=True,
#             dropout=dropout if num_layers > 1 else 0
#         )
#         self.attention = nn.Linear(hidden_size, 1)
#         self.dropout1 = nn.Dropout(0.5)
#         self.fc1 = nn.Linear(hidden_size, 32)
#         self.relu = nn.ReLU()
#         self.dropout2 = nn.Dropout(0.3)
#         self.fc2 = nn.Linear(32, 1)
#         self.sigmoid = nn.Sigmoid()

#     def forward(self, x):
#         gru_out, _ = self.gru(x)                                   # (batch, seq, hidden)
#         weights = torch.softmax(self.attention(gru_out), dim=1)    # (batch, seq, 1)
#         context = (gru_out * weights).sum(dim=1)                   # (batch, hidden)
#         out = self.dropout1(context)
#         out = self.fc1(out)
#         out = self.relu(out)
#         out = self.dropout2(out)
#         out = self.fc2(out)
#         out = self.sigmoid(out)
#         return out.squeeze(-1)


In [ ]:
class CNNBiLSTMModel(nn.Module):
    """Hybrid CNN + BiLSTM architecture"""
    def __init__(self, input_size=FEATURES, hidden_size=64, dropout=0.3):
        super(CNNBiLSTMModel, self).__init__()
        self.hidden_size = hidden_size

        # CNN block
        self.conv1 = nn.Conv1d(
            in_channels=input_size,
            out_channels=64,
            kernel_size=5,
            padding=2
        )
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            padding=1
        )
        self.relu2 = nn.ReLU()

        # BiLSTM block
        self.bilstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Fully connected
        self.dropout1 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_size * 2, 32)  # *2 because of bidirectional
        self.relu3 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch, SEQ_LEN=10, features=46)
        # English note.
        x = x.permute(0, 2, 1)  # (batch, 46, 80)

        # CNN block
        x = self.conv1(x)  # (batch, 64, 10)
        x = self.relu1(x)
        x = self.pool1(x)  # (batch, 64, 5)

        x = self.conv2(x)  # (batch, 128, 5)
        x = self.relu2(x)  # (batch, 128, 5)

        # Back to (batch, seq_len, channels) for LSTM
        x = x.permute(0, 2, 1)  # (batch, 5, 128)

        # BiLSTM block
        lstm_out, (h_n, c_n) = self.bilstm(x)
        last_hidden = lstm_out[:, -1, :]  # (batch, hidden_size*2)

        # FC block
        out = self.dropout1(last_hidden)
        out = self.fc1(out)
        out = self.relu3(out)
        out = self.dropout2(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out.squeeze(-1)


In [ ]:
# Parameter-counting function
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Create models
models_config = {
    'LSTM': LSTMModel(),
    'GRU': GRUModel(),
    'CNNBiLSTMModel': CNNBiLSTMModel()
}

print('Model architectures:\n')
for name, model in models_config.items():
    params = count_parameters(model)
    print(f'{name}: {params:,} parameters')

## Training and Evaluation Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """One training epoch"""
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss

In [ ]:
def validate_epoch(model, dataloader, criterion, device):
    """Model validation"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_loss += loss.item() * X_batch.size(0)
            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss, np.array(all_preds), np.array(all_labels)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, device, model_name, patience=10):
    """Full training loop with early stopping"""
    print('\nTraining model')

    model = model.to(device)
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_auc': []
    }

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    start_time = time.time()

    for epoch in range(num_epochs):
        # Training
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)

        # Validation
        val_loss, val_preds, val_labels = validate_epoch(model, val_loader, criterion, device)
        precision_p, recall_p, _ = precision_recall_curve(val_labels, val_preds)
        val_auc = auc(recall_p, precision_p)

        # Scheduler step
        scheduler.step(val_loss)

        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}] '
                  f'Train Loss: {train_loss:.4f} | '
                  f'Val Loss: {val_loss:.4f} | '
                  f'Val AUC: {val_auc:.4f} | '
                  f'LR: {optimizer.param_groups[0]["lr"]:.6f}')

        # Check early stopping
        if patience_counter >= patience:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

    training_time = time.time() - start_time
    print(f'\nTraining completed')
    print(f'Best Val Loss: {best_val_loss:.4f}')

    # English note.
    model.load_state_dict(best_model_state)

    return model, history, training_time

## Metric Calculation Functions

In [ ]:
def calculate_metrics(y_true, y_pred_proba, threshold=0.5):
    """Calculate all classification metrics"""
    y_pred = (y_pred_proba >= threshold).astype(int)

    # Base metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    avg_precision = average_precision_score(y_true, y_pred)
    precision_p, recall_p, _ = precision_recall_curve(y_true, y_pred)
    pr_auc = auc(recall_p, precision_p)
    roc_auc = roc_auc_score(y_true, y_pred)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # Additional metrics
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    metrics = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'F2-Score': f2,
        'AVG-Precision': avg_precision,
        'PR-AUC': pr_auc,
        'ROC-AUC': roc_auc,
        'Specificity': specificity,
        'FPR': fpr,
        'FNR': fnr,
        'TP': int(tp),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn)
    }

    return metrics

In [ ]:
def find_optimal_threshold(y_true, y_pred_proba, target_precision=0.99):
    """Search for the optimal threshold with target Precision"""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred_proba)

    # Find threshold for target precision
    idx = np.where(precisions >= target_precision)[0]
    if len(idx) > 0:
        optimal_idx = idx[np.argmax(recalls[idx])]
        optimal_threshold = thresholds[optimal_idx]
        optimal_precision = precisions[optimal_idx]
        optimal_recall = recalls[optimal_idx]
    else:
        # If target_precision cannot be reached, use the maximum
        optimal_idx = np.argmax(precisions)
        optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5
        optimal_precision = precisions[optimal_idx]
        optimal_recall = recalls[optimal_idx]

    return optimal_threshold, optimal_precision, optimal_recall

In [ ]:
def evaluate_model(model, dataloader, device, threshold=0.5):
    """Full model evaluation"""
    model.eval()
    all_preds = []
    all_labels = []
    inference_times = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)

            # Measure inference time
            start = time.time()
            outputs = model(X_batch)
            inference_times.append(time.time() - start)

            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(y_batch.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Calculate metrics
    metrics = calculate_metrics(all_labels, all_preds, threshold)

    # Average inference time per sample
    avg_inference_time = (sum(inference_times) / len(all_labels)) * 1000  # in ms
    metrics['Avg_Inference_Time_ms'] = avg_inference_time

    return metrics, all_preds, all_labels

In [ ]:
class WeightedBCELoss(nn.Module):
    """Custom weighted BCELoss to handle class imbalance."""
    def __init__(self, pos_weight):
        super().__init__()
        self.pos_weight = pos_weight

    def forward(self, pred, target):
        loss = -(self.pos_weight * target * torch.log(pred + 1e-7) +
                (1 - target) * torch.log(1 - pred + 1e-7))
        return loss.mean()

## Model Saving Functions

In [ ]:
def save_pytorch_checkpoint(model, model_name, history, metrics, metrics_opt, threshold, training_time, path):
    """Save weights and metadata to a .pth file."""
    safe_name = model_name.lower().replace("+", "_").replace(" ", "_")
    pth_filename = path / f'model_{safe_name}.pth'

    checkpoint = {
        'model_name': model_name,
        'model_state_dict': model.state_dict(),
        'model_architecture': type(model).__name__,
        'history': history,
        'test_metrics_default': metrics,
        'test_metrics_optimal': metrics_opt,
        'optimal_threshold': threshold,
        'training_time': training_time,
        'num_parameters': sum(p.numel() for p in model.parameters())
    }
    torch.save(checkpoint, pth_filename)
    print(f'  PyTorch checkpoint: {pth_filename.name}')

In [ ]:
def export_to_onnx(model, model_name, test_sample, path):
    """Export the model to ONNX with an accuracy check."""
    safe_name = model_name.lower().replace("+", "_").replace(" ", "_")
    onnx_filename = path / f'model_{safe_name}.onnx'

    model.eval()
    model_cpu = model.cpu()
    dummy_input = torch.randn(1, SEQ_LEN, FEATURES)  # Format (batch, seq, features)

    try:
        torch.onnx.export(
            model_cpu, dummy_input, onnx_filename,
            export_params=True, opset_version=18, do_constant_folding=True,
            input_names=['input'], output_names=['output'],
            dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
        )

        # Validation
        onnx_model = onnx.load(str(onnx_filename))
        onnx.checker.check_model(onnx_model)

        # Compare outputs (PyTorch vs ONNX)
        ort_session = ort.InferenceSession(str(onnx_filename))
        onnx_input = {ort_session.get_inputs()[0].name: test_sample.astype(np.float32)}
        onnx_output = ort_session.run(None, onnx_input)[0]

        with torch.no_grad():
            pt_output = model_cpu(torch.FloatTensor(test_sample)).numpy()

        diff = np.abs(pt_output - onnx_output).max()
        status = "OK" if diff < 1e-5 else "diff >= 1e-5"
        print(f'  {status} ONNX export completed (diff: {diff:.2e})')

    except Exception as e:
        print(f'  ONNX export error: {e}')

## Training All Models

In [ ]:
def run_model_experiment(model_name, model, config, loaders, device, data_path):
    """Full cycle: training, evaluation, saving."""
    print(f'\n # {model_name}\n')

    # Initialize components
    criterion = WeightedBCELoss(pos_weight=config['class_weights'][1])
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # Training
    trained_model, history, train_time = train_model(
        model=model, train_loader=loaders['train'], val_loader=loaders['val'],
        criterion=criterion, optimizer=optimizer, scheduler=scheduler,
        num_epochs=config['epochs'], device=device, model_name=model_name, patience=config['patience']
    )

    # Evaluation (Default & Optimal)
    test_m, test_p, test_l = evaluate_model(trained_model, loaders['test'], device, threshold=0.5)

    opt_threshold, _, _ = find_optimal_threshold(test_l, test_p, target_precision=0.99)
    test_m_opt, _, _ = evaluate_model(trained_model, loaders['test'], device, threshold=opt_threshold)

    # Save
    print(f'\nSave\n')
    save_pytorch_checkpoint(trained_model, model_name, history, test_m, test_m_opt, opt_threshold, train_time, data_path)

    # Use the first test sample for ONNX
    test_sample = next(iter(loaders['test']))[0][0:1].numpy()
    export_to_onnx(trained_model, model_name, test_sample, data_path)

    # Return to device
    trained_model.to(device)

    return {
        'model': trained_model,
        'history': history,
        'test_metrics_default': test_m,
        'test_metrics_optimal': test_m_opt,
        'optimal_threshold': opt_threshold,
        'test_preds': test_p,
        'test_labels': test_l
    }

In [ ]:
# Configuration
config = {
    'epochs': 30,
    'lr': 0.001,
    'patience': 10,
    'class_weights': class_weights # variable with weights
}

loaders = {'train': train_loader, 'val': val_loader, 'test': test_loader}
results = {}

for model_name, model in models_config.items():
    results[model_name] = run_model_experiment(model_name, model, config, loaders, device, DATA_PATH)

print('\n' + '='*80 + '\nALL MODELS TRAINED AND SAVED!\n' + '='*80)

## Comparative Model Analysis

In [ ]:
# Compare general characteristics
general_data = []
for model_name, res in results.items():
    metrics = res['test_metrics_default']
    general_data.append({
        'Model': model_name,
        'Specificity': metrics['Specificity'],
        'FPR': metrics['FPR'],
        'Inference Time (ms)': metrics['Avg_Inference_Time_ms'],
        'Parameters': count_parameters(res['model'])
    })

general_df = pd.DataFrame(general_data)
print('General characteristics')
print(general_df.to_string(index=False))
print('\n')

# English note.
comparison_data = []
for model_name, res in results.items():
    metrics = res['test_metrics_default']
    comparison_data.append({
        'Model': model_name,
        'Precision': metrics['Precision'],
        'Recall': metrics['Recall'],
        'F1-Score': metrics['F1-Score'],
        'F2-Score': metrics['F2-Score'],
        'ROC-AUC': metrics['ROC-AUC'],
        'PR-AUC': metrics['PR-AUC']
    })

comparison_df = pd.DataFrame(comparison_data)
print('Metrics (Test Set, threshold = 0.5)')
print(comparison_df.to_string(index=False))
print('\n')

# English note.
comparison_opt_data = []
for model_name, res in results.items():
    metrics = res['test_metrics_optimal']
    comparison_opt_data.append({
        'Model': model_name,
        'Precision': metrics['Precision'],
        'Recall': metrics['Recall'],
        'F1-Score': metrics['F1-Score'],
        'F2-Score': metrics['F2-Score'],
        'ROC-AUC': metrics['ROC-AUC'],
        'PR-AUC': metrics['PR-AUC'],
    })

comparison_opt_df = pd.DataFrame(comparison_opt_data)
print('Metrics with optimal threshold (Target Precision >= 0.99)')
print(comparison_opt_df.to_string(index=False))


## Result Visualization

In [ ]:
# Size
width = 15
hight = 5

# Training plots
fig, axes = plt.subplots(1, 3, figsize=(width, hight))
for idx, (model_name, res) in enumerate(results.items()):
    history = res['history']
    axes[idx].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[idx].plot(history['val_loss'], label='Val Loss', linewidth=2)
    axes[idx].set_xlabel('Epoch', fontsize=12)
    axes[idx].set_ylabel('Loss', fontsize=12)
    axes[idx].set_title(f'{model_name} - Training History', fontsize=14, fontweight='bold')
    axes[idx].legend(fontsize=11)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ROC Curves
colors = sns.color_palette("husl", len(results))
fig, axes = plt.subplots(1, 3, figsize=(width, hight))

for idx, (model_name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(res['test_labels'], res['test_preds'])
    auc_score = roc_auc_score(res['test_labels'], res['test_preds'])
    
    axes[idx].plot(fpr, tpr, label=f'{model_name} (AUC = {auc_score:.4f})', linewidth=2, color=colors[idx])
    axes[idx].set_xlabel('False Positive Rate', fontsize=14)
    axes[idx].set_ylabel('True Positive Rate (Recall)', fontsize=14)
    axes[idx].legend(fontsize=12)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('ROC Curves Comparison', fontsize=16, fontweight='bold') # Use suptitle for the common title
plt.tight_layout()
plt.tight_layout()
plt.show()

# Precision-Recall Curves
fig, axes = plt.subplots(1, 3, figsize=(width, hight))

for idx, (model_name, res) in enumerate(results.items()):
    precision, recall, _ = precision_recall_curve(res['test_labels'], res['test_preds'])
    pr_auc_score = average_precision_score(res['test_labels'], res['test_preds'])

    axes[idx].plot(recall, precision, label=f'{model_name} (PR-AUC = {pr_auc_score:.4f})', linewidth=2, color=colors[idx])
    axes[idx].set_xlabel('Recall', fontsize=14)
    axes[idx].set_ylabel('Precision', fontsize=14)
    axes[idx].legend(fontsize=12)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Precision-Recall Curves Comparison', fontsize=16, fontweight='bold') # Use suptitle for the common title
plt.tight_layout()
plt.tight_layout()
plt.show()

# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(width, hight))
for idx, (model_name, res) in enumerate(results.items()):
    metrics = res['test_metrics_default']
    cm = np.array([[metrics['TN'], metrics['FP']],
                   [metrics['FN'], metrics['TP']]])

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Fault'],
                yticklabels=['Normal', 'Fault'],
                ax=axes[idx], cbar=True, annot_kws={"size": 14})
    axes[idx].set_xlabel('Predicted', fontsize=12)
    axes[idx].set_ylabel('Actual', fontsize=12)
    axes[idx].set_title(f'{model_name} - Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Metric comparison
fig, axes = plt.subplots(1, 3, figsize=(width, hight))
metrics_to_plot = ['PR-AUC', 'ROC-AUC', 'Avg_Inference_Time_ms']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    values = [results[model]['test_metrics_default'][metric] for model in results.keys()]
    bars = ax.bar(results.keys(), values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f'{metric}', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    # # Add values to bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height/ 2, f'{val:.4f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()